# Coverage, connectivity and topology enrichment

Reproduces the corresponding Streamlit plots. Choose the sidebar inputs below. Calculations use the existing canonical networks and UniProt mappings. Expensive optional simulations are disabled by default.

In [1]:
from pathlib import Path
import os, sys, time
root = Path.cwd()
if not (root / "moonlighting.py").exists(): root = root.parent
os.chdir(root)
sys.path.insert(0, str(root))
import numpy as np
import polars as pl
import plotly.express as px
try:
    from IPython.display import display
except ImportError:
    display = print
from networks import discover_datasets
from moonlighting import (load_moon_data, cohort_ids, graph_from_path, matching_nodes,
    multi_network_centrality_comparison, centrality_heatmap, cross_network_metric_figure,
    hub_enrichment_heatmap, shortest_path_monte_carlo, subset_betweenness_analysis,
    bh_adjust, significance_label)

## Parameters and data

Set these to match the Streamlit sidebar. Advanced calculations exclude networks with more than 500,000 edges.

In [2]:
sources = ['moonlitdb']  # None = all sources
set_name = 'MoonLitDB'
require_all, reviewed_only, min_publications, seed = False, False, 0, 42
runs = 100
run_shortest_path_simulation = False
run_subset_betweenness = False
run_subset_null = False
benchmark_exact = False  # True bypasses centrality caches
available = discover_datasets(Path('data'), isoforms='collapse')
network_names = [n for n, p in available.items() if p.parent.name != 'versions' and (n != 'huMAP2' or 'huMAP2_0.05' not in available)]
network_names += [n for n in ('huMAP2_0.05', 'huMAP2_0.95') if n in available and n not in network_names]
moon = load_moon_data()
cohort, selected = cohort_ids(moon, sources, require_all, reviewed_only, min_publications)
assert cohort and network_names, 'Select networks and a nonempty mapped protein set.'
graphs = {n: graph_from_path(available[n]) for n in network_names}
eligible = {n: g for n, g in graphs.items() if g['edge_count'] <= 500_000}
print('Protein set:', set_name, '| mapped accessions:', len(cohort))
print('Inputs:', {n: str(available[n]) for n in network_names})
print('Excluded from advanced analysis (>500,000 edges):', list(set(graphs) - set(eligible)))

Protein set: MoonLitDB | mapped accessions: 458
Inputs: {'HuRI': 'data\\processed_networks\\HuRI.parquet', 'BioPlex': 'data\\processed_networks\\BioPlex.parquet', 'TAGC_binary2025': 'data\\processed_networks\\TAGC_binary2025.parquet', 'huMAP2_0.05': 'data\\processed_networks\\versions\\huMAP2_0.05.parquet', 'huMAP2_0.95': 'data\\processed_networks\\versions\\huMAP2_0.95.parquet'}
Excluded from advanced analysis (>500,000 edges): []


## Coverage

The denominator is the mapped protein-set accessions, including those absent from a network. This is descriptive; no test applies.

In [3]:
rows = []
for name, graph in graphs.items():
    nodes = set(map(str, graph['nodes']))
    labels = {label for node in nodes for label in node.split('|')}
    present = matching_nodes(nodes, cohort)
    n, e = len(nodes), graph['edge_count']
    covered = len(cohort & labels)
    rows.append({'Network': name, 'Nodes': n, 'Edges': e, 'Mean degree': 2*e/n if n else 0,
        'Density': 2*e/(n*(n-1)) if n > 1 else 0, f'{set_name} proteins present': covered,
        f'{set_name} coverage (%)': 100*covered/len(cohort), f'{set_name} network nodes': len(present),
        'Background nodes': n-len(present)})
stats = pl.DataFrame(rows)
display(stats)
fig = px.bar(stats, x='Network', y=f'{set_name} coverage (%)', text=f'{set_name} proteins present', color='Network', title=f'Share of {set_name} present in each interactome')
fig.update_layout(template='plotly_white', showlegend=False, height=380)
fig.show()

Network,Nodes,Edges,Mean degree,Density,MoonLitDB proteins present,MoonLitDB coverage (%),MoonLitDB network nodes,Background nodes
str,i64,i64,f64,f64,i64,f64,i64,i64
"""HuRI""",8163,51574,12.636041,0.001548,205,44.759825,205,7958
"""BioPlex""",13834,128104,18.520168,0.001339,394,86.026201,394,13440
"""TAGC_binary2025""",14235,108649,15.265051,0.001072,431,94.104803,431,13804
"""huMAP2_0.05""",8709,25713,5.904926,0.000678,330,72.052402,330,8379
"""huMAP2_0.95""",3273,6619,4.044607,0.001236,161,35.152838,161,3112


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

## Shortest-path Monte Carlo

Degree-matched random sets are sampled from the network complement; shortest paths can use all nodes. The two-sided empirical test uses +1 correction and BH across tested networks. Enable the switch above to run.

In [ ]:
if run_shortest_path_simulation:
    simulation_networks = list(eligible)[:1]  # change to match Streamlit
    mc = {n: shortest_path_monte_carlo(eligible[n], cohort, runs, seed, True) for n in simulation_networks}
    qs = dict(zip(mc, bh_adjust([r['summary']['p_value'] for r in mc.values()])))
    for name, result in mc.items():
        s = result['summary']
        fig = px.histogram(result['null'].filter(pl.col('mean_shortest_path').is_finite()), x='mean_shortest_path', nbins=35, title=f'{name}: {set_name} shortest-path null distribution')
        if s['observed'] is not None and np.isfinite(s['observed']): fig.add_vline(x=s['observed'], line_color='firebrick')
        fig.add_annotation(x=0, y=1.12, xref='paper', yref='paper', xanchor='left', showarrow=False, text=f"Effect {s['effect_observed_minus_null']} steps · BH q={qs[name]:.3g} {significance_label(qs[name])}")
        fig.update_layout(template='plotly_white', margin=dict(t=120), yaxis_title='Simulations')
        fig.show()
        display(pl.DataFrame([s]))

## Subset betweenness

Endpoints are protein-set members and intermediate nodes may be any network protein. Node scores are descriptive. The optional aggregate test uses degree-matched complement sets with BH across tested networks.

In [ ]:
if run_subset_betweenness:
    subset_networks = list(eligible)[:1]
    subset = {n: subset_betweenness_analysis(eligible[n], cohort, runs if run_subset_null else 0, seed, True) for n in subset_networks}
    qs = dict(zip(subset, bh_adjust([r['summary']['p_value'] for r in subset.values()]))) if run_subset_null else {}
    for name, result in subset.items():
        scores = result['scores'].filter(pl.col('in_protein_set')).sort(['subset_betweenness', 'protein'], descending=[True, False]).head(25)
        fig = px.bar(scores, x='subset_betweenness', y='protein', orientation='h', title=f'{name}: {set_name} internal mediators (top 25)')
        fig.update_layout(template='plotly_white', height=max(350, 22*scores.height+120), yaxis=dict(autorange='reversed'))
        fig.show()
        if run_subset_null:
            s = result['summary']
            fig = px.histogram(result['null'].filter(pl.col('mean_internal_set_intermediaries').is_finite()), x='mean_internal_set_intermediaries', nbins=35, title=f'{name}: null distribution of within-set mediation')
            if s['mean_internal_set_intermediaries'] is not None: fig.add_vline(x=s['mean_internal_set_intermediaries'], line_color='firebrick')
            fig.add_annotation(x=0, y=1.12, xref='paper', yref='paper', xanchor='left', showarrow=False, text=f"Effect {s['effect_observed_minus_null']} · BH q={qs[name]:.3g} {significance_label(qs[name])}")
            fig.update_layout(template='plotly_white', margin=dict(t=120), yaxis_title='Simulations')
            fig.show()
        display(pl.DataFrame([result['summary']]))

## Topology and hub enrichment

Exact unweighted betweenness uses every network protein as a source. Each protein set is compared with all other nodes in the same network. Mann–Whitney tests use BH across all network-by-metric comparisons; one-sided hypergeometric hub tests have a separate BH family. Hubs are at or above the ties-inclusive 90th percentile. Stars: * q < 0.05, ** q < 0.01, *** q < 0.001, **** q < 0.0001; ns means non-significant and n/a unavailable. Topology alone does not validate moonlighting or binding exclusivity.

In [ ]:
assert eligible, 'No selected networks satisfy the 500,000-edge guard.'
start = time.perf_counter()
result = multi_network_centrality_comparison(eligible, cohort, extra=True, seed=seed, landmarks=None, use_cache=not benchmark_exact, set_name=set_name)
print(f"Overview elapsed: {time.perf_counter()-start:.2f} seconds; cache enabled: {not benchmark_exact}")
print('Fresh overview timing includes all centralities and statistical comparisons, not just betweenness.')
metrics = ['degree', 'betweenness_approx', 'core_number', 'clustering']
profiles = result['profiles'].filter(pl.col('metric').is_in(metrics))
centrality_heatmap(profiles, annotation_mode='q').show()
for metric in metrics:
    cross_network_metric_figure(result['metrics'], metric, result['profiles'], plot_kind='box', log_scale=(metric == 'degree'), set_name=set_name, annotation_mode='q').show()
hub_enrichment_heatmap(result['hub_enrichment']).show()
display(result['profiles'])
display(result['hub_enrichment'])